# NASA Battery B0005 — Parallel CNN-LSTM optimized with GA and Hyperband (Option C)

This notebook optimizes the **parallel** CNN-LSTM on the Battery dataset using the same two
optimizers (GA and Hyperband) that were applied to the **sequential** model, so both
architectures are compared under equal optimization on this dataset.

Pipeline matches the reported Battery experiments: capacity signal, sliding window of 10
cycles, MinMax scaling, train on cycles < 50 and test on cycles >= 50 of B0005, RMSE
computed on capacity. Baseline parallel test RMSE = 0.047.

In [1]:
%matplotlib inline
import os, random
import numpy as np, pandas as pd
from scipy.io import loadmat
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, LSTM, Dense, Dropout, Flatten, Concatenate
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from deap import base, creator, tools, algorithms
import keras_tuner as kt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

In [2]:
# ---- load B0005 discharge capacity (this notebook sits in the battery dataset folder) ----
MAT_PATH = 'B0005.mat'

def load_capacity(battery, path):
    mat = loadmat(path)
    rows = []
    c = 0
    for i in range(len(mat[battery][0,0]['cycle'][0])):
        r = mat[battery][0,0]['cycle'][0,i]
        if r['type'][0] == 'discharge':
            c += 1
            rows.append([c, r['data'][0][0]['Capacity'][0][0]])
    return pd.DataFrame(rows, columns=['cycle','capacity'])

cap = load_capacity('B0005', MAT_PATH)
print('cycles', len(cap))

cycles 168


In [3]:
WINDOW = 10
train_df = cap[cap['cycle'] < 50]
test_df  = cap[cap['cycle'] >= 50]

sc = MinMaxScaler(feature_range=(0,1))
train_scaled = sc.fit_transform(train_df[['capacity']].values)

X, y = [], []
for i in range(WINDOW, len(train_scaled)):
    X.append(train_scaled[i-WINDOW:i, 0]); y.append(train_scaled[i, 0])
X = np.array(X).reshape(-1, WINDOW, 1); y = np.array(y)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=SEED)

# rolling test windows over the full trajectory
full = np.concatenate([train_df['capacity'].values, test_df['capacity'].values])
inp = sc.transform(full[len(full)-len(test_df)-WINDOW:].reshape(-1,1))
X_te = np.array([inp[i-WINDOW:i,0] for i in range(WINDOW, len(inp))]).reshape(-1, WINDOW, 1)
y_te = test_df['capacity'].values
print('X_tr', X_tr.shape, 'X_te', X_te.shape)

def rmse_cap(model):
    p = sc.inverse_transform(model.predict([X_te, X_te], verbose=0))[:,0]
    return float(np.sqrt(mean_squared_error(y_te, p)))

X_tr (31, 10, 1) X_te (119, 10, 1)


In [4]:
def build_parallel(filters=64, kernel=3, n_conv=2, units=100, n_lstm=2,
                   dropout=0.0, lr=1e-3, optimizer='adam'):
    """Two-branch parallel CNN-LSTM (matches the Battery parallel baseline family)."""
    in_cnn = Input(shape=(WINDOW, 1)); in_lstm = Input(shape=(WINDOW, 1))
    c = in_cnn
    for _ in range(n_conv):
        c = Conv1D(filters, kernel_size=kernel, activation='relu', padding='same')(c)
    c = Flatten()(c)
    l = in_lstm
    for i in range(n_lstm):
        l = LSTM(units, return_sequences=(i < n_lstm-1))(l)
    m = Concatenate()([c, l])
    if dropout > 0: m = Dropout(dropout)(m)
    out = Dense(1, activation='linear')(m)
    model = Model([in_cnn, in_lstm], out)
    opt = Adam(lr) if optimizer == 'adam' else RMSprop(lr)
    model.compile(loss='mean_squared_error', optimizer=opt)
    return model

## Genetic Algorithm (DEAP)

In [5]:
filters_choices = [32, 64, 96, 128]
kernel_choices  = [3, 5]
units_choices   = [32, 64, 100, 128]
dropout_choices = [0.0, 0.2, 0.3]
lr_choices      = list(np.round(np.linspace(1e-4, 1e-2, 10), 6))

if 'FitnessMinB' not in dir(creator):
    creator.create('FitnessMinB', base.Fitness, weights=(-1.0,))
    creator.create('IndividualB', list, fitness=creator.FitnessMinB)

tb = base.Toolbox()
tb.register('f',  np.random.randint, 0, len(filters_choices))
tb.register('k',  np.random.randint, 0, len(kernel_choices))
tb.register('nc', np.random.randint, 1, 3)
tb.register('u',  np.random.randint, 0, len(units_choices))
tb.register('nl', np.random.randint, 1, 3)
tb.register('dr', np.random.randint, 0, len(dropout_choices))
tb.register('lr', np.random.randint, 0, len(lr_choices))
tb.register('individual', tools.initCycle, creator.IndividualB,
            (tb.f, tb.k, tb.nc, tb.u, tb.nl, tb.dr, tb.lr), n=1)
tb.register('population', tools.initRepeat, list, tb.individual)

def evaluate(ind):
    try:
        m = build_parallel(filters=filters_choices[ind[0]], kernel=kernel_choices[ind[1]],
                           n_conv=ind[2], units=units_choices[ind[3]], n_lstm=ind[4],
                           dropout=dropout_choices[ind[5]], lr=lr_choices[ind[6]])
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        m.fit([X_tr, X_tr], y_tr, validation_data=([X_va, X_va], y_va),
              epochs=30, batch_size=16, callbacks=[es], verbose=0)
        return (float(np.sqrt(mean_squared_error(y_va, m.predict([X_va,X_va], verbose=0)[:,0]))),)
    except Exception as e:
        print('eval failed:', e); return (1e5,)

tb.register('evaluate', evaluate)
tb.register('mate', tools.cxTwoPoint)
tb.register('mutate', tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
tb.register('select', tools.selTournament, tournsize=3)

In [6]:
POP, NGEN = 10, 5
pop = tb.population(n=POP)
bounds = [len(filters_choices), len(kernel_choices), None, len(units_choices), None,
          len(dropout_choices), len(lr_choices)]
for gen in range(NGEN):
    print('Generation', gen+1)
    off = algorithms.varAnd(pop, tb, cxpb=0.5, mutpb=0.3)
    for ind in off:
        for j,b in enumerate(bounds):
            if b is None: ind[j]=int(np.clip(round(ind[j]),1,2))
            else:         ind[j]=int(np.clip(round(ind[j]),0,b-1))
    for ind, fit in zip(off, map(tb.evaluate, off)):
        ind.fitness.values = fit
    pop = tb.select(off, k=len(pop))
best = tools.selBest(pop, 1)[0]
print('Best genes', list(best), 'val RMSE', best.fitness.values[0])

Generation 1
Generation 2
Generation 3
Generation 4
Generation 5
Best genes [3, 0, 2, 3, 2, 0, 4] val RMSE 0.09491389759766368


In [7]:
ga_model = build_parallel(filters=filters_choices[best[0]], kernel=kernel_choices[best[1]],
                          n_conv=best[2], units=units_choices[best[3]], n_lstm=best[4],
                          dropout=dropout_choices[best[5]], lr=lr_choices[best[6]])
ga_model.fit([X_tr, X_tr], y_tr, validation_data=([X_va, X_va], y_va),
             epochs=100, batch_size=16,
             callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
             verbose=1)
ga_rmse = rmse_cap(ga_model)
print('GA parallel test RMSE:', round(ga_rmse, 4))

Epoch 1/100
2/2 [==============================] - 3s 664ms/step - loss: 0.3302 - val_loss: 0.2616
Epoch 2/100
2/2 [==============================] - 0s 66ms/step - loss: 0.1122 - val_loss: 0.0251
Epoch 3/100
2/2 [==============================] - 0s 61ms/step - loss: 0.0825 - val_loss: 0.0180
Epoch 4/100
2/2 [==============================] - 0s 52ms/step - loss: 0.0375 - val_loss: 0.0791
Epoch 5/100
2/2 [==============================] - 0s 55ms/step - loss: 0.0522 - val_loss: 0.0744
Epoch 6/100
2/2 [==============================] - 0s 51ms/step - loss: 0.0340 - val_loss: 0.0224
Epoch 7/100
2/2 [==============================] - 0s 62ms/step - loss: 0.0361 - val_loss: 0.0167
Epoch 8/100
2/2 [==============================] - 0s 56ms/step - loss: 0.0393 - val_loss: 0.0194
Epoch 9/100
2/2 [==============================] - 0s 58ms/step - loss: 0.0270 - val_loss: 0.0440
Epoch 10/100
2/2 [==============================] - 0s 50ms/step - loss: 0.0318 - val_loss: 0.0527
Epoch 11/100
2/2 [

## Hyperband (Keras Tuner)

Same search scope Hyperband was given for the Battery sequential model — CNN filters,
LSTM units, learning rate — factor 3, max_epochs 10.

In [8]:
def hb_build(hp):
    f  = hp.Int('cnn_filters', 32, 128, step=32)
    u  = hp.Int('lstm_units', 32, 128, step=32)
    lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    return build_parallel(filters=f, kernel=3, n_conv=2, units=u, n_lstm=2, lr=lr)

tuner = kt.Hyperband(hb_build, objective='val_loss', max_epochs=10, factor=3,
                     directory='hb_battery_par', project_name='par_optionC', overwrite=True)
tuner.search([X_tr, X_tr], y_tr, validation_data=([X_va, X_va], y_va),
             epochs=10, batch_size=16,
             callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)])
hb_model = tuner.get_best_models(1)[0]
hb_model.fit([X_tr, X_tr], y_tr, validation_data=([X_va, X_va], y_va),
             epochs=100, batch_size=16,
             callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
             verbose=1)
hb_rmse = rmse_cap(hb_model)
print('Hyperband parallel test RMSE:', round(hb_rmse, 4))

Trial 30 Complete [00h 00m 05s]
val_loss: 0.14120256900787354

Best val_loss So Far: 0.01220451109111309
Total elapsed time: 00h 02m 06s
Epoch 1/100
2/2 [==============================] - 3s 675ms/step - loss: 0.0195 - val_loss: 0.0137
Epoch 2/100
2/2 [==============================] - 0s 49ms/step - loss: 0.0157 - val_loss: 0.0074
Epoch 3/100
2/2 [==============================] - 0s 48ms/step - loss: 0.0154 - val_loss: 0.0214
Epoch 4/100
2/2 [==============================] - 0s 49ms/step - loss: 0.0178 - val_loss: 0.0100
Epoch 5/100
2/2 [==============================] - 0s 54ms/step - loss: 0.0143 - val_loss: 0.0090
Epoch 6/100
2/2 [==============================] - 0s 52ms/step - loss: 0.0164 - val_loss: 0.0199
Epoch 7/100
2/2 [==============================] - 0s 53ms/step - loss: 0.0133 - val_loss: 0.0142
Epoch 8/100
2/2 [==============================] - 0s 51ms/step - loss: 0.0129 - val_loss: 0.0165
Epoch 9/100
2/2 [==============================] - 0s 52ms/step - loss: 0.0132

In [9]:
import json
summary = {'dataset':'NASA Battery B0005','architecture':'parallel','window':WINDOW,
           'baseline_test_rmse':0.047,
           'ga_test_rmse':round(ga_rmse,4),
           'hyperband_test_rmse':round(hb_rmse,4)}
print(summary)
os.makedirs('output', exist_ok=True)
json.dump(summary, open('output/battery_b0005_parallel_optionC.json','w'), indent=2)

{'dataset': 'NASA Battery B0005', 'architecture': 'parallel', 'window': 10, 'baseline_test_rmse': 0.047, 'ga_test_rmse': 0.5269, 'hyperband_test_rmse': 0.4917}
